# Notebook 4 — Entrenamiento de Modelo NER Local
## EpiDiagnostix-Mayab — Componente Extractor Clínico (ML Supervisado)

**Objetivo:** Entrenar un modelo propio de Reconocimiento de Entidades Nombradas (NER) 
usando spaCy + Machine Learning supervisado. El resultado es un archivo binario local 
que funciona 100% offline, sin APIs de pago ni servidores externos.

---

## ¿Por qué NER supervisado y no regex?

| | Regex | **NER Supervisado (este notebook)** |
|---|---|---|
| Aprende del dato | ❌ Nunca | ✅ Aprende de miles de ejemplos |
| "peso de 78 kg" | Requiere regla explícita | ✅ Lo infiere del contexto |
| "78 kilos" | Requiere otra regla | ✅ Mismo modelo |
| Offline | ✅ | ✅ |
| Gratuito | ✅ | ✅ |
| Necesita API | ❌ | ❌ |
| Generalización | ❌ Frágil | ✅ Aprende variantes no vistas |

## Arquitectura del modelo

```
Texto libre
    ↓
Tokenizador spaCy (español)
    ↓
Tok2Vec  ←── capa de embeddings (representa contexto)
    ↓
NER transition-based parser
    ↓
Entidades etiquetadas [(inicio, fin, ETIQUETA), ...]
    ↓
Post-procesador (texto → valor numérico/categoría)
```

## 11 entidades a extraer

| Etiqueta | Variable | Tipo |
|---|---|---|
| `EDAD` | Edad del paciente | int (años) |
| `SEXO` | Sexo del paciente | str (M/F) |
| `PESO_KG` | Peso | float (kg) |
| `TALLA_CM` | Talla | float (cm) |
| `PRESION_SIS` | Presión sistólica | int (mmHg) |
| `PRESION_DIA` | Presión diastólica | int (mmHg) |
| `GLUCOSA` | Glucosa | int (mg/dL) |
| `TEMPERATURA` | Temperatura | float (°C) |
| `FREC_CARD` | Frecuencia cardíaca | int (lpm) |
| `DURACION` | Duración de síntomas | int (días) |
| `CATEGORIA` | Categoría clínica | str |

---
## Paso 1 — Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import random
import json
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
from spacy.scorer import Scorer

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH   = '../consultas_clinicas.csv'
MODELS_DIR  = Path('../models')
MODEL_PATH  = MODELS_DIR / 'ner_clinico_es'
MODELS_DIR.mkdir(exist_ok=True)

# Hiperparámetros de entrenamiento
N_EPOCHS   = 60
BATCH_SIZE = 32
DROPOUT    = 0.35
VAL_SPLIT  = 0.10
TEST_SPLIT = 0.10

print(f'spaCy versión: {spacy.__version__}')
print(f'Configuración: {N_EPOCHS} epochs, batch={BATCH_SIZE}, dropout={DROPOUT}')

---
## Paso 2 — Definición del formato de entrenamiento

### ¿Qué es el formato NER de spaCy?

spaCy necesita ejemplos de entrenamiento en este formato:
```python
("texto de ejemplo", {"entities": [(inicio, fin, "ETIQUETA")]})
```

Donde `inicio` y `fin` son las posiciones de caracteres de la entidad en el texto.

**Ejemplo:**
```
"Paciente masculino de 45 años"
 0123456789012345678901234567890
          ^         ^
          9         18  → SEXO: "masculino"
                    22  25 → EDAD: "45"
```

In [ ]:
# Todas las etiquetas de entidades
ETIQUETAS = [
    'EDAD', 'SEXO', 'PESO_KG', 'TALLA_CM',
    'PRESION_SIS', 'PRESION_DIA',
    'GLUCOSA', 'TEMPERATURA', 'FREC_CARD',
    'DURACION', 'CATEGORIA'
]

# Función central: construye texto fragmento a fragmento y registra posiciones
def hacer_texto(*partes):
    """
    Recibe pares (texto_fragmento, etiqueta_o_None).
    Construye el texto completo y calcula las posiciones exactas de caracteres.
    
    Ejemplo:
        hacer_texto(
            ("Paciente ", None),
            ("masculino", "SEXO"),
            (" de ", None),
            ("45", "EDAD"),
            (" años.", None)
        )
        → ("Paciente masculino de 45 años.",
           [(9, 18, 'SEXO'), (22, 24, 'EDAD')])
    """
    texto_final = ""
    entidades   = []
    for (texto, etiqueta) in partes:
        if etiqueta and texto.strip():
            inicio = len(texto_final)
            fin    = inicio + len(texto)
            entidades.append((inicio, fin, etiqueta))
        texto_final += texto
    return texto_final, entidades


# Verificación con un ejemplo
t, e = hacer_texto(
    ("Paciente ",    None),
    ("masculino",    "SEXO"),
    (" de ",         None),
    ("45",           "EDAD"),
    (" años, peso ", None),
    ("82.0",         "PESO_KG"),
    (" kg.",         None),
)
print("Texto:", t)
print("Entidades:")
for inicio, fin, etiqueta in e:
    print(f"  [{inicio}:{fin}] '{t[inicio:fin]}' → {etiqueta}")

---
## Paso 3 — Generación del Dataset

### ¿Por qué generar datos sintéticos?

No tenemos las transcripciones de audio originales, solo las variables ya estructuradas.
Generamos ~28,000 textos médicos con múltiples estilos de redacción para que el modelo
aprenda a extraer variables independientemente del fraseo.

### Estilos de redacción (10 variantes por fila)

In [ ]:
# Vocabularios de variantes por campo

SEXO_M = ['masculino', 'hombre', 'varón', 'del sexo masculino', 'paciente masculino']
SEXO_F = ['femenino', 'mujer', 'femenina', 'del sexo femenino', 'paciente femenina']
SEXO_M_COLOQUIAL = ['señor', 'el señor', 'niño', 'varón']
SEXO_F_COLOQUIAL = ['señora', 'la señora', 'niña', 'paciente']

PREFIJOS_EDAD = ['de {v} años', '{v} años de edad', '{v} años cumplidos',
                 'edad: {v} años', '{v} años', 'edad {v}']

PREFIJOS_PESO = ['{v} kg', '{v} kilogramos', '{v} kilos', 'peso {v} kg',
                 'pesa {v} kg', 'peso de {v} kg', 'peso corporal: {v} kg',
                 '{v} kg de peso', 'masa corporal {v} kg']

PREFIJOS_TALLA = ['{v} cm', '{v} centímetros', 'talla {v} cm',
                  'mide {v} cm', 'estatura de {v} cm', 'altura {v} cm',
                  'talla: {v} centímetros']

PREFIJOS_PA   = ['presión arterial {s}/{d} mmHg', 'presión {s}/{d}',
                 'PA: {s}/{d}', 'tensión {s}/{d} mmHg',
                 'presión arterial de {s} sobre {d}',
                 '{s} sobre {d} de presión', 'tensión arterial {s}/{d}']

PREFIJOS_GLUCOSA = ['glucosa {v} mg/dL', 'glucosa de {v} mg/dl',
                    'glucemia {v}', 'glucosa en ayunas {v}',
                    'glucosa en ayunas de {v} mg', 'nivel de glucosa {v}',
                    'azúcar en sangre {v}', 'azúcar {v} mg/dl',
                    'glucosa capilar {v} mg/dL']

PREFIJOS_TEMP = ['temperatura {v}°C', 'temperatura {v} grados',
                 'temperatura de {v} grados', 'temperatura corporal {v}°C',
                 'T: {v} grados', '{v}°C', 'fiebre de {v} grados',
                 'temperatura {v}', 'temperatura normal {v}',
                 'temperatura de {v}°C']

PREFIJOS_FC   = ['frecuencia cardiaca {v} lpm', 'FC: {v} lpm',
                 'pulso {v} lpm', 'frecuencia cardiaca de {v} latidos por minuto',
                 'ritmo cardiaco {v}', '{v} latidos por minuto',
                 'frecuencia cardíaca {v} lpm', 'pulso {v} latidos']

PREFIJOS_DUR  = ['{v} días de evolución', 'desde hace {v} días',
                 'lleva {v} días', 'cuadro de {v} días',
                 'evolución de {v} días', '{v} días con síntomas',
                 '{v} días de cuadro clínico']

INTRO_CAT     = ['categoría: ', 'categoría clínica: ', 'impresión: ',
                 'cuadro ', 'diagnóstico orientado a ', 'se orienta a ',
                 'categoría clínica de ']

print(f'Variantes de peso: {len(PREFIJOS_PESO)}')
print(f'Variantes de glucosa: {len(PREFIJOS_GLUCOSA)}')
print(f'Variantes de temperatura: {len(PREFIJOS_TEMP)}')
print('Vocabularios de variantes cargados.')

In [ ]:
def fmt(v, decimales=1):
    """Formatea número como string, quitando .0 si es entero."""
    if isinstance(v, float) and v == int(v):
        return str(int(v))
    return str(round(v, decimales))


def metros(cm):
    """Convierte cm a string en metros con 2 decimales."""
    return str(round(cm / 100, 2))


def escoger(lista, rng):
    return rng.choice(lista)


# ────────────────────────────────────────────────────────────────────────────
# ESTILOS DE GENERACIÓN (10 estilos distintos)
# ────────────────────────────────────────────────────────────────────────────

def estilo_soap(r, rng):
    """Estilo 1 — SOAP formal con todas las entidades."""
    sexo_v = escoger(SEXO_M if r.sexo=='M' else SEXO_F, rng)
    cat    = r.categoria_sintoma.lower()
    intro  = escoger(INTRO_CAT, rng)
    return hacer_texto(
        ("Consulta médica. Paciente ", None),
        (sexo_v, "SEXO"),
        (", ", None),
        (fmt(r.edad), "EDAD"),
        (" años. Peso ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kg, talla ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" cm. Presión arterial ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (" mmHg. Glucosa ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (" mg/dL. Temperatura ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        ("°C. FC ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (" lpm. Evolución ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días. ", None),
        (intro, None),
        (r.categoria_sintoma, "CATEGORIA"),
        (".", None),
    )


def estilo_dictado_rapido(r, rng):
    """Estilo 2 — Dictado rápido con abreviaciones."""
    sexo_v = escoger(SEXO_M if r.sexo=='M' else SEXO_F, rng)
    return hacer_texto(
        ("Px ", None),
        (sexo_v, "SEXO"),
        (", ", None),
        (fmt(r.edad), "EDAD"),
        (" años. PA: ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (". Glucemia ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (". T: ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (" grados. FC: ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (" lpm. Peso: ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kg. Talla: ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" cm. Evo: ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días.", None),
    )


def estilo_narrativo(r, rng):
    """Estilo 3 — Narrativo fluido."""
    sexo_v  = escoger(SEXO_M_COLOQUIAL if r.sexo=='M' else SEXO_F_COLOQUIAL, rng)
    edad_str = escoger(PREFIJOS_EDAD, rng).format(v=fmt(r.edad))
    intro_cat = escoger(INTRO_CAT, rng)
    return hacer_texto(
        ("Se atiende a ", None),
        (sexo_v, "SEXO"),
        (" ", None),
        (fmt(r.edad), "EDAD"),
        (" años. Peso ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kg, talla ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" cm. Presión ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        (" sobre ", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (". Azúcar en sangre ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (". Temperatura ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (". Pulso ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (". Lleva ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días con síntomas. ", None),
        (intro_cat, None),
        (r.categoria_sintoma, "CATEGORIA"),
        (".", None),
    )


def estilo_coloquial(r, rng):
    """Estilo 4 — Coloquial, como se habla en comunidades de Chiapas."""
    sexo_v = escoger(SEXO_M_COLOQUIAL if r.sexo=='M' else SEXO_F_COLOQUIAL, rng)
    return hacer_texto(
        ("La ", None) if r.sexo == 'F' else ("El ", None),
        (sexo_v, "SEXO"),
        (" tiene como ", None),
        (fmt(r.edad), "EDAD"),
        (" años. Presión le salió ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        (" sobre ", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (". El azúcar ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (". Temperatura normal ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (". Pulso ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (". Pesa como ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kilos, mide ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" centímetros. Lleva ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días así.", None),
    )


def estilo_talla_metros(r, rng):
    """Estilo 5 — Talla expresada en metros (caso crítico para generalización)."""
    sexo_v = escoger(SEXO_M if r.sexo=='M' else SEXO_F, rng)
    talla_m = metros(r.talla_cm)
    intro_cat = escoger(INTRO_CAT, rng)
    return hacer_texto(
        ("Paciente ", None),
        (sexo_v, "SEXO"),
        (" de ", None),
        (fmt(r.edad), "EDAD"),
        (" años. Peso ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kg, talla de ", None),
        (talla_m, "TALLA_CM"),         # valor en metros — post-procesador convierte
        (" m. Presión arterial de ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (" mmHg. Glucosa en ayunas de ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (" mg. Temperatura ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (" grados. Frecuencia cardíaca de ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (" latidos por minuto. Evolución de ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días. ", None),
        (intro_cat, None),
        (r.categoria_sintoma, "CATEGORIA"),
        (".", None),
    )


def estilo_solo_signos_vitales(r, rng):
    """Estilo 6 — Solo signos vitales (texto parcial, útil para robustez)."""
    return hacer_texto(
        ("Signos vitales: presión ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (", glucosa ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (" mg/dl, temperatura ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        ("°C, FC ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (" lpm.", None),
    )


def estilo_orden_alterado(r, rng):
    """Estilo 7 — Orden diferente (prueba generalización posicional)."""
    sexo_v = escoger(SEXO_M if r.sexo=='M' else SEXO_F, rng)
    return hacer_texto(
        ("PA ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (". Glucosa ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (". Temperatura ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (". FC ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (". Paciente ", None),
        (sexo_v, "SEXO"),
        (" de ", None),
        (fmt(r.edad), "EDAD"),
        (" años. ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kg, ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" cm. Consulta por ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días de síntomas.", None),
    )


def estilo_con_variantes_vocabulario(r, rng):
    """Estilo 8 — Vocabulario variado (glucemia, tensión, etc.)."""
    sexo_v    = escoger(SEXO_M if r.sexo=='M' else SEXO_F, rng)
    gluc_pre  = escoger(PREFIJOS_GLUCOSA, rng)
    temp_pre  = escoger(PREFIJOS_TEMP, rng)
    fc_pre    = escoger(PREFIJOS_FC, rng)
    dur_pre   = escoger(PREFIJOS_DUR, rng)
    # Solo usar el patrón con la parte de texto antes del valor
    gluc_pre_str = gluc_pre.split('{v}')[0]
    gluc_suf_str = gluc_pre.split('{v}')[1] if '{v}' in gluc_pre else ''
    temp_pre_str = temp_pre.split('{v}')[0]
    temp_suf_str = temp_pre.split('{v}')[1] if '{v}' in temp_pre else ''
    return hacer_texto(
        ("Paciente ", None),
        (sexo_v, "SEXO"),
        (", ", None),
        (fmt(r.edad), "EDAD"),
        (" años. Tensión ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (". ", None),
        (gluc_pre_str, None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (gluc_suf_str + ". ", None),
        (temp_pre_str, None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (temp_suf_str + ". ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (" latidos. Peso ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kg. Talla ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" cm. ", None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días de cuadro.", None),
    )


def estilo_solo_datos_basicos(r, rng):
    """Estilo 9 — Solo datos demográficos y antropométricos."""
    sexo_v = escoger(SEXO_M if r.sexo=='M' else SEXO_F, rng)
    return hacer_texto(
        ("Paciente ", None),
        (sexo_v, "SEXO"),
        (", edad ", None),
        (fmt(r.edad), "EDAD"),
        (" años. Peso: ", None),
        (fmt(r.peso_kg), "PESO_KG"),
        (" kilogramos. Estatura: ", None),
        (fmt(r.talla_cm), "TALLA_CM"),
        (" centímetros.", None),
    )


def estilo_categoria_sintoma(r, rng):
    """Estilo 10 — Énfasis en categoría clínica y duración."""
    sexo_v    = escoger(SEXO_M_COLOQUIAL if r.sexo=='M' else SEXO_F_COLOQUIAL, rng)
    intro_cat = escoger(INTRO_CAT, rng)
    dur_pre   = escoger(PREFIJOS_DUR, rng).split('{v}')[0]
    return hacer_texto(
        ("Consulta de ", None),
        (sexo_v, "SEXO"),
        (" de ", None),
        (fmt(r.edad), "EDAD"),
        (" años. ", None),
        (dur_pre, None),
        (str(int(r.duracion_sintomas_dias)), "DURACION"),
        (" días con síntomas. Presión ", None),
        (str(int(r.presion_sistolica)), "PRESION_SIS"),
        ("/", None),
        (str(int(r.presion_diastolica)), "PRESION_DIA"),
        (". Glucosa ", None),
        (str(int(r.glucosa_mg_dl)), "GLUCOSA"),
        (". Temperatura ", None),
        (fmt(r.temperatura_c), "TEMPERATURA"),
        (". FC ", None),
        (str(int(r.frecuencia_cardiaca_bpm)), "FREC_CARD"),
        (". ", None),
        (intro_cat, None),
        (r.categoria_sintoma, "CATEGORIA"),
        (".", None),
    )


ESTILOS = [
    estilo_soap,
    estilo_dictado_rapido,
    estilo_narrativo,
    estilo_coloquial,
    estilo_talla_metros,
    estilo_solo_signos_vitales,
    estilo_orden_alterado,
    estilo_con_variantes_vocabulario,
    estilo_solo_datos_basicos,
    estilo_categoria_sintoma,
]

print(f'{len(ESTILOS)} estilos de redacción definidos.')

In [ ]:
# Cargar datos y generar el dataset completo
df = pd.read_csv(DATA_PATH)
rng = random.Random(SEED)
DATASET = []

for _, row in df.iterrows():
    for estilo_fn in ESTILOS:
        try:
            texto, entidades = estilo_fn(row, rng)
            if texto and entidades:  # descartar si salió vacío
                DATASET.append((texto, {"entities": entidades}))
        except Exception:
            pass   # silenciar errores de filas con datos faltantes

print(f'Dataset generado: {len(DATASET):,} ejemplos')
print(f'  → {len(df):,} filas × {len(ESTILOS)} estilos')
print()
print('Ejemplo aleatorio:')
t, ann = rng.choice(DATASET)
print(f'  Texto: {t}')
print(f'  Entidades:')
for ini, fin, etiq in ann['entities']:
    print(f'    [{ini}:{fin}] "{t[ini:fin]}" → {etiq}')

---
## Paso 4 — Estadísticas del Dataset

In [ ]:
from collections import Counter

conteos = Counter()
for _, ann in DATASET:
    for _, _, etiq in ann['entities']:
        conteos[etiq] += 1

print('Conteo de entidades en el dataset:')
total_ents = sum(conteos.values())
for etiq in ETIQUETAS:
    n = conteos.get(etiq, 0)
    print(f'  {etiq:<15} {n:>7,}  ({n/total_ents*100:.1f}%)')
print(f'  ─────────────────────────────')
print(f'  TOTAL          {total_ents:>7,}')

fig, ax = plt.subplots(figsize=(10, 4))
etqs = [e for e in ETIQUETAS if e in conteos]
vals = [conteos[e] for e in etqs]
ax.bar(etqs, vals, color='steelblue')
ax.set_title('Distribución de entidades en el dataset de entrenamiento')
ax.set_ylabel('Cantidad de ejemplos')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/ner_dataset_distribucion.png', bbox_inches='tight')
plt.show()

---
## Paso 5 — División Train / Validación / Test

En Machine Learning supervisado siempre dividimos los datos en tres partes:
- **Train (80%):** el modelo aprende con estos datos.
- **Validación (10%):** monitoreamos el aprendizaje durante el entrenamiento. No se usa para actualizar pesos.
- **Test (10%):** evaluación final, nunca visto durante el entrenamiento.

In [ ]:
rng.shuffle(DATASET)
n = len(DATASET)
n_test = int(n * TEST_SPLIT)
n_val  = int(n * VAL_SPLIT)
n_train = n - n_test - n_val

train_data = DATASET[:n_train]
val_data   = DATASET[n_train:n_train+n_val]
test_data  = DATASET[n_train+n_val:]

print(f'Split del dataset:')
print(f'  Train      : {len(train_data):>7,} ({len(train_data)/n*100:.0f}%)')
print(f'  Validación : {len(val_data):>7,} ({len(val_data)/n*100:.0f}%)')
print(f'  Test       : {len(test_data):>7,} ({len(test_data)/n*100:.0f}%)')
print(f'  Total      : {n:>7,}')

---
## Paso 6 — Construcción del Modelo spaCy

### ¿Cómo funciona el entrenamiento de NER?

```
Texto tokenizado → Tok2Vec → representación vectorial de cada token
                                        ↓
                            NER Transition-based parser
                            (aprende cuándo ABRIR/CERRAR una entidad)
                                        ↓
                            Predicción: [(inicio, fin, ETIQUETA)]
                                        ↓
                            Comparar con la verdad (loss)
                                        ↓
                            Backpropagation → actualizar pesos
```

El proceso se repite por cada ejemplo en cada época. Con **60 épocas × 28K ejemplos**,
el modelo actualiza sus pesos ~1.7 millones de veces.

In [ ]:
# Crear modelo spaCy en blanco para español
# Usamos blanco porque queremos que aprenda NUESTRAS entidades clínicas,
# no las de un modelo preentrenado en noticias.
nlp = spacy.blank("es")

# Agregar el componente NER al pipeline
ner = nlp.add_pipe("ner", last=True)

# Registrar las 11 etiquetas
for etiq in ETIQUETAS:
    ner.add_label(etiq)

print('Pipeline del modelo:')
print(f'  Componentes: {nlp.pipe_names}')
print(f'  Etiquetas NER: {ner.labels}')
print(f'  Idioma base: español ("es")')

---
## Paso 7 — Bucle de Entrenamiento (Training Loop)

Este es el corazón del Machine Learning. Por cada época:
1. **Mezclar** los datos de entrenamiento (evita que el modelo aprenda el orden)
2. **Mini-batches**: procesar grupos de ejemplos juntos (más eficiente que uno a uno)
3. **Forward pass**: el modelo predice las entidades
4. **Calcular loss**: qué tan lejos está la predicción de la verdad
5. **Backward pass**: ajustar los pesos para reducir el error (backpropagation)
6. **Evaluar** en el set de validación para detectar overfitting

In [ ]:
from spacy.training import Example
from spacy.util import minibatch

def hacer_examples(nlp, datos):
    """Convierte pares (texto, anotaciones) en objetos Example de spaCy."""
    examples = []
    for texto, anotaciones in datos:
        doc = nlp.make_doc(texto)
        try:
            ex = Example.from_dict(doc, anotaciones)
            examples.append(ex)
        except Exception:
            pass   # descartar si hay problema de alineación de tokens
    return examples


def evaluar(nlp, datos_eval):
    """Evalúa el modelo en el set de validación. Devuelve F1 promedio."""
    scorer  = Scorer()
    examples = hacer_examples(nlp, datos_eval)
    scores   = nlp.evaluate(examples)
    return scores.get("ents_f", 0.0), scores.get("ents_p", 0.0), scores.get("ents_r", 0.0)


# Inicializar optimizer
optimizer = nlp.begin_training()
optimizer.learn_rate = 0.001

historial = {'epoch': [], 'loss': [], 'val_f1': [], 'val_p': [], 'val_r': []}
mejor_f1  = 0.0
paciencia  = 0
MAX_PACIENCIA = 10  # early stopping

print(f'Iniciando entrenamiento: {N_EPOCHS} épocas, batch={BATCH_SIZE}, dropout={DROPOUT}')
print(f'Ejemplos de entrenamiento: {len(train_data):,}')
print()

for epoch in range(N_EPOCHS):
    # 1. Mezclar datos
    random.shuffle(train_data)

    # 2. Mini-batch training
    losses = {}
    batches = minibatch(train_data, size=BATCH_SIZE)
    for batch in batches:
        examples = hacer_examples(nlp, batch)
        if examples:
            nlp.update(examples, sgd=optimizer, drop=DROPOUT, losses=losses)

    # 3. Evaluar en validación cada 5 épocas
    loss_ner = losses.get('ner', 0.0)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        val_f1, val_p, val_r = evaluar(nlp, val_data)
        historial['epoch'].append(epoch + 1)
        historial['loss'].append(loss_ner)
        historial['val_f1'].append(val_f1)
        historial['val_p'].append(val_p)
        historial['val_r'].append(val_r)
        print(f'Época {epoch+1:>3}/{N_EPOCHS}  |  Loss: {loss_ner:8.2f}  |  Val F1: {val_f1:.4f}  P: {val_p:.4f}  R: {val_r:.4f}')

        # Early stopping
        if val_f1 > mejor_f1:
            mejor_f1 = val_f1
            paciencia = 0
            nlp.to_disk(MODEL_PATH)  # guardar el mejor
        else:
            paciencia += 1
            if paciencia >= MAX_PACIENCIA:
                print(f'\nEarly stopping en época {epoch+1} — mejor F1: {mejor_f1:.4f}')
                break

print(f'\nEntrenamiento completo. Mejor F1 validación: {mejor_f1:.4f}')
print(f'Modelo guardado en: {MODEL_PATH}')

---
## Paso 8 — Visualización del Aprendizaje

Estas curvas muestran cómo el modelo fue mejorando con cada época.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(historial['epoch'], historial['loss'], 'o-', color='crimson', linewidth=2)
ax.set_xlabel('Época')
ax.set_ylabel('Loss NER')
ax.set_title('Curva de pérdida durante el entrenamiento\n(↓ más bajo = mejor aprendizaje)')
ax.grid(True)

ax2 = axes[1]
ax2.plot(historial['epoch'], historial['val_f1'], 'o-', color='seagreen', linewidth=2, label='F1')
ax2.plot(historial['epoch'], historial['val_p'],  's--', color='steelblue', linewidth=1.5, label='Precisión')
ax2.plot(historial['epoch'], historial['val_r'],  '^--', color='darkorange', linewidth=1.5, label='Recall')
ax2.set_xlabel('Época')
ax2.set_ylabel('Score (0-1)')
ax2.set_title('Métricas en validación por época\n(↑ más alto = mejor generalización)')
ax2.legend()
ax2.set_ylim(0, 1.05)
ax2.grid(True)

plt.suptitle('Curvas de Entrenamiento — NER Clínico EpiDiagnostix-Mayab', fontsize=13)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/ner_curvas_entrenamiento.png', bbox_inches='tight')
plt.show()
print('Curvas guardadas.')

---
## Paso 9 — Evaluación Final en el Set de Test

El test set es datos que el modelo **nunca vio** durante el entrenamiento ni la validación.
Esta es la medida real de qué tan bien generaliza a texto médico nuevo.

In [ ]:
# Cargar el mejor modelo guardado
nlp_final = spacy.load(MODEL_PATH)

# Evaluar en test
test_examples = hacer_examples(nlp_final, test_data)
scores_test   = nlp_final.evaluate(test_examples)

print('=== EVALUACIÓN FINAL EN TEST SET ===')
print(f'  F1  global: {scores_test["ents_f"]:.4f}')
print(f'  Precisión : {scores_test["ents_p"]:.4f}')
print(f'  Recall    : {scores_test["ents_r"]:.4f}')
print()

# F1 por entidad
print('  F1 por entidad:')
print(f'  {"Entidad":<18} {"F1":>8} {"Precisión":>10} {"Recall":>8}')
print('  ' + '-'*48)
ents_scores = scores_test.get('ents_per_type', {})
for etiq in ETIQUETAS:
    if etiq in ents_scores:
        s   = ents_scores[etiq]
        f1  = s.get('f', 0)
        p   = s.get('p', 0)
        r   = s.get('r', 0)
        estado = 'EXCELENTE' if f1 >= 0.85 else 'BUENO' if f1 >= 0.70 else 'MEJORAR'
        print(f'  {etiq:<18} {f1:>8.4f} {p:>10.4f} {r:>8.4f}  {estado}')
    else:
        print(f'  {etiq:<18} {"N/A":>8}')

In [ ]:
# Visualización de F1 por entidad
etqs_vis = [e for e in ETIQUETAS if e in ents_scores]
f1_vals  = [ents_scores[e].get('f', 0) for e in etqs_vis]
orden    = sorted(range(len(f1_vals)), key=lambda i: f1_vals[i])
etqs_ord = [etqs_vis[i] for i in orden]
f1_ord   = [f1_vals[i] for i in orden]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['seagreen' if v >= 0.85 else 'steelblue' if v >= 0.70 else 'crimson' for v in f1_ord]
bars   = ax.barh(etqs_ord, f1_ord, color=colors)
ax.axvline(0.85, color='seagreen', linestyle='--', alpha=0.6, label='Umbral excelente (0.85)')
ax.axvline(0.70, color='steelblue', linestyle='--', alpha=0.6, label='Umbral bueno (0.70)')
for bar, val in zip(bars, f1_ord):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('F1-Score')
ax.set_title('F1 por entidad clínica — NER local entrenado\n(verde=excelente, azul=bueno, rojo=mejorar)')
ax.legend(fontsize=8)
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/ner_f1_por_entidad.png', bbox_inches='tight')
plt.show()

---
## Paso 10 — Post-procesador: texto → valor estructurado

El modelo NER extrae **texto** ("masculino", "78.0", "1.72"). El post-procesador
lo convierte al **tipo correcto** ("M", 78.0, 172.0).

In [ ]:
import unicodedata

SEXO_MAP = {
    'masculino': 'M', 'hombre': 'M', 'varon': 'M', 'nino': 'M',
    'senor': 'M', 'senores': 'M',
    'femenino': 'F', 'mujer': 'F', 'femenina': 'F', 'nina': 'F',
    'senora': 'F', 'paciente': None,  # ambiguo
}

CAT_NORMALIZE = {
    'gastrointestinal': 'Gastrointestinal',
    'respiratorio':     'Respiratorio',
    'hipertension':     'Hipertensión',
    'diabetes':         'Diabetes',
    'vacunacion':       'Vacunación',
    'nutricion':        'Nutrición',
    'embarazo':         'Embarazo',
    'traumatologia':    'Traumatología',
    'dermatologico':    'Dermatológico',
    'infeccioso':       'Infeccioso/Vectorial',
    'vectorial':        'Infeccioso/Vectorial',
    'dengue':           'Infeccioso/Vectorial',
}


def norm(s):
    """Quita tildes y pasa a minúsculas."""
    return unicodedata.normalize('NFD', str(s)).encode('ascii', 'ignore').decode().lower().strip()


def post_procesar(doc, texto_original):
    """Convierte las entidades reconocidas en el diccionario de variables clínicas."""
    resultado = {e: None for e in [
        'edad', 'sexo', 'peso_kg', 'talla_cm',
        'presion_sistolica', 'presion_diastolica',
        'glucosa_mg_dl', 'temperatura_c',
        'frecuencia_cardiaca_bpm', 'duracion_sintomas_dias',
        'categoria_sintoma'
    ]}

    for ent in doc.ents:
        txt = ent.text.strip()
        try:
            if ent.label_ == 'EDAD':
                resultado['edad'] = int(float(txt))

            elif ent.label_ == 'SEXO':
                resultado['sexo'] = SEXO_MAP.get(norm(txt))

            elif ent.label_ == 'PESO_KG':
                resultado['peso_kg'] = round(float(txt.replace(',','.')), 1)

            elif ent.label_ == 'TALLA_CM':
                v = float(txt.replace(',','.'))
                if v < 3.0:          # estaba en metros
                    v = round(v * 100, 1)
                resultado['talla_cm'] = round(v, 1)

            elif ent.label_ == 'PRESION_SIS':
                resultado['presion_sistolica'] = int(float(txt))

            elif ent.label_ == 'PRESION_DIA':
                resultado['presion_diastolica'] = int(float(txt))

            elif ent.label_ == 'GLUCOSA':
                resultado['glucosa_mg_dl'] = int(float(txt))

            elif ent.label_ == 'TEMPERATURA':
                resultado['temperatura_c'] = round(float(txt.replace(',','.')), 1)

            elif ent.label_ == 'FREC_CARD':
                resultado['frecuencia_cardiaca_bpm'] = int(float(txt))

            elif ent.label_ == 'DURACION':
                resultado['duracion_sintomas_dias'] = int(float(txt))

            elif ent.label_ == 'CATEGORIA':
                k = norm(txt)
                resultado['categoria_sintoma'] = CAT_NORMALIZE.get(k, txt)

        except (ValueError, TypeError):
            pass

    return resultado


print('Post-procesador definido.')

---
## Paso 11 — Prueba de Inferencia con Textos Reales

Probamos el modelo con los mismos 5 casos que usamos en el extractor regex,
para comparar directamente ambos enfoques.

In [ ]:
CASOS_PRUEBA = [
    {
        "nombre": "Caso 1 — SOAP clásico",
        "texto": "Consulta médica. Paciente masculino de 45 años de edad. "
                 "Peso 82.0 kg, talla 170.5 cm. "
                 "Presión arterial de 125/82 mmHg, glucosa de 98 mg/dl, "
                 "temperatura 37.2°C, frecuencia cardiaca de 91 latidos por minuto. "
                 "5 días de evolución. Categoría: Gastrointestinal.",
        "esperado": {"edad":45,"sexo":"M","peso_kg":82.0,"talla_cm":170.5,
                     "presion_sistolica":125,"presion_diastolica":82,
                     "glucosa_mg_dl":98,"temperatura_c":37.2,
                     "frecuencia_cardiaca_bpm":91,"duracion_sintomas_dias":5}
    },
    {
        "nombre": "Caso 2 — Dictado rápido",
        "texto": "Paciente femenina, 67 años. PA: 160/95, glucemia capilar 310 mg/dL. "
                 "T: 36.8 grados. FC: 88 lpm. Peso 63.5 kg, estatura de 155.0 cm. "
                 "Síntomas desde hace 3 días. Categoría: Diabetes.",
        "esperado": {"edad":67,"sexo":"F","presion_sistolica":160,
                     "presion_diastolica":95,"glucosa_mg_dl":310,"duracion_sintomas_dias":3}
    },
    {
        "nombre": "Caso 3 — Coloquial comunidad",
        "texto": "Se atiende a mujer de 32 años que lleva 7 días con tos. "
                 "Presión 118 sobre 74, azúcar en sangre 89, temperatura 38.5, pulso 102. "
                 "Pesa 58 kilos, mide 162 cm. Categoría: Respiratorio.",
        "esperado": {"edad":32,"sexo":"F","presion_sistolica":118,
                     "glucosa_mg_dl":89,"temperatura_c":38.5,"duracion_sintomas_dias":7}
    },
    {
        "nombre": "Caso 4 — Talla en metros (crítico)",
        "texto": "Paciente masculino de 42 años con peso de 78 kg y talla de 1.72 m. "
                 "Presión arterial de 120 sobre 80. Glucosa en ayunas de 95 mg. "
                 "Temperatura 38.5 grados. Frecuencia cardíaca de 85 latidos por minuto. "
                 "4 días de evolución. Dengue sospechoso.",
        "esperado": {"edad":42,"sexo":"M","peso_kg":78.0,"talla_cm":172.0,
                     "presion_sistolica":120,"glucosa_mg_dl":95,"temperatura_c":38.5,
                     "frecuencia_cardiaca_bpm":85,"duracion_sintomas_dias":4}
    },
    {
        "nombre": "Caso 5 — Texto libre sin estructura",
        "texto": "La señora tiene como 55 años, es hipertensa, visión borrosa "
                 "desde hace 4 días. Presión 170 sobre 100. Azúcar estaba bien, 95. "
                 "Temperatura normal 36.5. Pulso 78. "
                 "Pesa como 70 kilos y mide 158 centímetros.",
        "esperado": {"edad":55,"presion_sistolica":170,"presion_diastolica":100,
                     "glucosa_mg_dl":95,"temperatura_c":36.5,"duracion_sintomas_dias":4}
    },
]

TOLERANCIAS_NUM = {
    'edad': 1, 'peso_kg': 1.0, 'talla_cm': 2.0,
    'presion_sistolica': 1, 'presion_diastolica': 1,
    'glucosa_mg_dl': 1, 'temperatura_c': 0.2,
    'frecuencia_cardiaca_bpm': 1, 'duracion_sintomas_dias': 1,
}

total_ok = 0
total_campos = 0

for caso in CASOS_PRUEBA:
    doc    = nlp_final(caso['texto'])
    result = post_procesar(doc, caso['texto'])
    esp    = caso['esperado']

    caso_ok = 0
    print(f"\n{'='*60}")
    print(f"  {caso['nombre']}")
    print(f"  Texto: {caso['texto'][:90]}...")
    print(f"  Entidades detectadas: {[(e.text, e.label_) for e in doc.ents]}")
    print()
    print(f"  {'Campo':<28} {'Extraído':>12}  {'Esperado':>12}  OK?")
    print("  " + "-"*58)

    for campo, val_esp in esp.items():
        val_pred = result.get(campo)
        tol = TOLERANCIAS_NUM.get(campo)
        if tol and val_pred is not None:
            ok = abs(float(val_pred) - float(val_esp)) <= tol
        else:
            ok = str(val_pred).lower() == str(val_esp).lower()
        marca = 'OK' if ok else 'FALLA'
        print(f"  {campo:<28} {str(val_pred):>12}  {str(val_esp):>12}  {marca}")
        total_campos += 1
        if ok:
            caso_ok += 1
            total_ok += 1

    pct = caso_ok / len(esp) * 100 if esp else 0
    print(f"\n  Resultado: {caso_ok}/{len(esp)} ({pct:.0f}%)")

print(f"\n{'='*60}")
print(f"  RESULTADO GLOBAL: {total_ok}/{total_campos} ({total_ok/total_campos*100:.1f}%)")
print(f"{'='*60}")

---
## Paso 12 — Exportar modelo para producción

El modelo se guarda como un directorio binario de spaCy.
Para cargarlo en producción (en la clínica, sin internet):
```python
import spacy
nlp = spacy.load("models/ner_clinico_es")
doc = nlp("Paciente de 45 años, presión 120/80...")
```

In [ ]:
# El mejor modelo ya fue guardado durante el entrenamiento con early stopping.
# Verificamos el tamaño del directorio.
import os

def dir_size_mb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 * 1024)

size_mb = dir_size_mb(MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')
print(f'Tamaño en disco:    {size_mb:.1f} MB')
print()

# Listar archivos del modelo
print('Archivos del modelo:')
for root, dirs, files in os.walk(MODEL_PATH):
    nivel = root.replace(str(MODEL_PATH), '').count(os.sep)
    indent = '  ' * nivel
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp) / 1024
        print(f'{indent}  {f} ({sz:.0f} KB)')

# Verificar carga
nlp_verificacion = spacy.load(MODEL_PATH)
doc_test = nlp_verificacion("Paciente masculino de 45 años, presión 125/82.")
print()
print('Verificación de carga:')
print(f'  Texto: "{doc_test.text}"')
print(f'  Entidades: {[(e.text, e.label_) for e in doc_test.ents]}')

---
## Paso 13 — Resumen para la tesis

In [ ]:
f1_global = scores_test['ents_f']
p_global  = scores_test['ents_p']
r_global  = scores_test['ents_r']

print('=' * 65)
print('  RESUMEN — Modelo NER Local EpiDiagnostix-Mayab')
print('=' * 65)
print()
print('  Tipo:          NER supervisado con spaCy')
print('  Idioma:        Español (modelo base en blanco)')
print('  Internet:      NO REQUIERE — 100% offline')
print('  Costo:         $0.00 — libre de APIs de pago')
print()
print('  Dataset:')
print(f'    Fuente: consultas_clinicas.csv ({len(df):,} filas)')
print(f'    Estilos: {len(ESTILOS)} variantes de redacción por fila')
print(f'    Total ejemplos: {len(DATASET):,}')
print(f'    Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}')
print()
print('  Entrenamiento:')
print(f'    Épocas: {N_EPOCHS} (con early stopping)')
print(f'    Batch size: {BATCH_SIZE}')
print(f'    Dropout: {DROPOUT}')
print()
print('  Métricas en TEST SET (datos nunca vistos):')
print(f'    F1  global: {f1_global:.4f}')
print(f'    Precisión : {p_global:.4f}')
print(f'    Recall    : {r_global:.4f}')
print()
print('  F1 por entidad:')
for etiq in ETIQUETAS:
    s  = ents_scores.get(etiq, {})
    f1 = s.get('f', 0)
    print(f'    {etiq:<18} {f1:.4f}')
print()
print(f'  Tamaño del modelo: {size_mb:.1f} MB (adecuado para cómputos de bajos recursos)')
print(f'  Ubicación: {MODEL_PATH}')
print()
print('  Ventaja clave sobre regex:')
print('    El modelo APRENDE del contexto. "peso de 78 kg", "pesa 78",')
print('    "78 kilos", "masa 78 kg" → todos reconocidos sin una sola regla adicional.')
print()
print('  Siguientes pasos:')
print('    1. Integrar como adapter en el microservicio hexagonal')
print('    2. Ampliar dataset con transcripciones reales de audio')
print('    3. Fine-tuning con errores de dictado detectados en producción')
print('=' * 65)